In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls benchmarks infra scripts config.yaml

In [ ]:
# ONE-TIME: copy the CSVs of the categories in config.yaml from Drive to the bucket (skips ones already there)
import yaml
CATS = yaml.safe_load(open("config.yaml"))["data"]["categories"]
print("categories:", CATS)
for cat in CATS:
    !gcloud storage cp --no-clobber /content/drive/MyDrive/amazon_reviews_2018/ratings_only/{cat}.csv gs://YOUR_BUCKET/raw/amazon_reviews_2018/ratings_only/
!gcloud storage ls -l gs://YOUR_BUCKET/raw/amazon_reviews_2018/ratings_only/

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_launch.sh 16

In [ ]:
# ONE-TIME per set of categories: convert them to Parquet (untimed). Skip if Test C or the pipeline run already did it.
!zip -q -r /tmp/code.zip etl centralized evaluation federated benchmarks
!gcloud storage cp config.yaml gs://YOUR_BUCKET/full/config.yaml
P = "spark.dynamicAllocation.enabled=false,spark.executor.instances=16,spark.executor.cores=3,spark.executor.memory=9g"
!gcloud dataproc jobs submit pyspark scripts/run_etl.py --cluster=da231o --region=asia-south1 --py-files=/tmp/code.zip --properties={P} -- --config=gs://YOUR_BUCKET/full/config.yaml --step=convert 2>&1 | grep -E "^\[etl|Traceback|Error|state:"

In [ ]:
!python scripts/run_benchmarks.py --config config.yaml --modes etl

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh